# 05 — One-stage anchor-free: YOLO11

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import settings

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import comparison_tables, completed_runs
from vod.variants import collect_view, yolo_detector
from vod.yolo import P2_CFG, YOLO, evaluate_yolo, train_yolo, write_data_yaml, write_dataset

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
TMP = Path("/tmp/yolo_data") if Path("/kaggle/working").exists() else WORK / "yolo_data"
ROOTS = (WORK, Path("/kaggle/input"))
settings.update({"wandb": False})

## Data
Same frames as the other detectors. Ignored regions are masked, val only for monitoring: final numbers come
from vod.evaluate on the original frames.

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
val_images = write_dataset(val_frames.iloc[::PROTOCOL["val_step_epoch"]], val_targets, TMP / "val")


def data_yaml(degrade_p):
    images = write_dataset(train_frames, train_targets, TMP / f"train_d{int(degrade_p * 100)}", degrade_p=degrade_p)
    return write_data_yaml(TMP / f"data_d{int(degrade_p * 100)}.yaml", images, val_images)

## Experiments

In [ ]:
BASE = dict(weights="yolo11s.pt", imgsz=PROTOCOL["imgsz"], epochs=12, batch=8, close_mosaic=3, degrade_p=0.0)
LOW_LR = dict(optimizer="SGD", lr0=0.002, cos_lr=True)
EXPERIMENTS = {
    "yolo_masked": BASE,
    "yolo_lowlr": {**BASE, **LOW_LR},
    "yolo_cls1": {**BASE, **LOW_LR, "cls": 1.0},
    "yolo_strong": {**BASE, **LOW_LR, "degrade_p": 0.3, "scale": 0.9, "mixup": 0.1, "hsv_v": 0.6},
    "yolo_p2": {**BASE, **LOW_LR, "model": P2_CFG, "batch": 4},
    "yolo_m": {**BASE, **LOW_LR, "weights": "yolo11m.pt", "batch": 4},
}
RUN = list(EXPERIMENTS)

## Train and evaluate (last epoch, every val frame)

In [ ]:
for name in RUN:
    done = completed_runs(ROOTS)
    if name in done:
        print(name, "done")
        continue
    cfg = EXPERIMENTS[name]
    print(name, "training")
    run_dir = train_yolo(name, cfg, data_yaml(cfg["degrade_p"]), WORK)
    m = evaluate_yolo(name, cfg, run_dir, val_frames, VAL_GT)
    print({k: round(m[k], 4) for k in ["mAP", "AP50", "AP75", "AP_small"]}, m["ms_per_image"], "ms/image")
    torch.cuda.empty_cache()

## Comparison

In [ ]:
runs = completed_runs(ROOTS)
main, per_class, per_clip = comparison_tables(runs, CLASSES, names=EXPERIMENTS)
display(main)
display(per_class)
per_clip

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if n in EXPERIMENTS and m.get("history") is not None:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (every 10th frame)", title="Validation mAP during training")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK / "yolo_map_curves.png", dpi=110)

## Final model: best config retrained on train + val
No val number exists for this model, only the leaderboard. Set FINAL after reading the comparison.

In [ ]:
FINAL = None  # e.g. "yolo_p2"

In [ ]:
if FINAL:
    cfg = EXPERIMENTS[FINAL]
    d = cfg["degrade_p"]
    train_images = write_dataset(train_frames, train_targets, TMP / f"train_d{int(d * 100)}", degrade_p=d)
    extra = write_dataset(val_frames.iloc[::PROTOCOL["train_step"]], val_targets, TMP / f"valtrain_d{int(d * 100)}", degrade_p=d)
    yaml_path = write_data_yaml(TMP / "data_final.yaml", train_images + extra, val_images)
    run_dir = train_yolo(f"{FINAL}_final", cfg, yaml_path, WORK)
    test = pd.read_csv(DATA / "test_images.csv")
    test_paths = [str(DATA / f) for f in test["file_name"]]
    dets = collect_view(yolo_detector(YOLO(run_dir / "weights" / "last.pt"), imgsz=cfg["imgsz"]), test_paths, "full")
    dets["image_id"] = dets.pop("file_name").map(dict(zip(test_paths, test["image_id"])))
    sub = write_submission(dets, DATA / "test_images.csv", WORK / f"submission_{FINAL}_final.csv")
    display(sub.head())

In [ ]:
shutil.rmtree(TMP, ignore_errors=True)